<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec13_binary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec13_binary.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第13回 二値画像処理
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib

MATLAB のライブスクリプト `code/matlab/vie_sec13_binary.m` を Python に移植したもの。第13回のスライド（vie2026-13）で使う図と数値を計算する。閾値処理と大津法，ハーフトーニング（パターンニング，ディザリング，誤差拡散），カラー画像への応用，順序統計フィルタとモルフォロジー処理を扱う。記号は教科書に合わせる（閾値 $\tau$，度数 $h_x$，クラス間分散 $\sigma_\mathrm{B}^2$，構造化要素 $\mathsf{h}$）。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

画像処理には scikit-image，SciPy，NumPy を使う。教科書の例題の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import os
import urllib.request

if not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

import math

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.draw import polygon2mask
from skimage.filters import threshold_otsu
from skimage.transform import downscale_local_mean
from skimage.util import img_as_float
import vie

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

写真は参考資料のサンプル画像 msipimg08（スイカ，参考資料の図 2.5 と同じ画像）をグレースケールにし，$256\times256$ に縮小して使う。縞模様のスイカが暗い背景の中にあり，閾値の選び方の違い（階調の中央と大津法）がはっきり見える。

In [ ]:
X = vie.msipimg(8, 256, "gray").astype(np.float64)   # 8 bit（0〜255），256×256

## 閾値処理（$\tau=128$）
各画素値を閾値と比較して白（1）か黒（0）にする。$\tau=128$ は 8 bit の階調の中央。画素値が整数なので，教科書の例「二値化閾値処理」の $\tau=x_\mathrm{max}/2=127.5$ と同じ結果になる。

In [ ]:
print(np.array_equal(X >= 128, X >= 127.5))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6), layout="constrained")
show(axs[0], X / 255, "原画像")
show(axs[1], (X >= 128).astype(float), r"閾値処理（$\tau=128$）")
plt.show()

## 大津法の手順（教科書 2.2.3 項）
大津法は，黒クラス（$i=0$，$x<\tau$）と白クラス（$i=1$，$x\geq\tau$）のクラス間分散

$$\sigma_\mathrm{B}^2(\tau)=\frac{1}{N^2}n_0(\tau)n_1(\tau)\bigl(\mu_0(\tau)-\mu_1(\tau)\bigr)^2$$

を最大にする閾値 $\tau^\star$ を選ぶ。次の関数 `otsu` は，教科書の手順どおり度数 $h_x$ から画素数 $n_i(\tau)$ と画素値の累積和 $c_i(\tau)$ を $\tau=1,2,\ldots,L-1$ の順に更新し，$n_i(\tau)=0$ のときの平均値 $\mu_i(\tau)$ を NaN（数値として定義できない値）とする。

注意：scikit-image の `threshold_otsu` が返す閾値は黒クラスに含まれる最大の画素値を表す。教科書の $\tau^\star$（白クラスに含まれる最小の画素値）とは $\tau^\star=\texttt{threshold\_otsu}+1$ の関係にある。

In [ ]:
def otsu(x, L):
    """教科書 2.2.3 項の手順による大津法（x は 0〜L-1 の整数）．

    tauStar : クラス間分散を最大にする閾値（x < tauStar が黒，x >= tauStar が白）
    tbl     : τ=0,1,...,L-1 に対する h_τ, n0, n1, mu0, mu1, sigmaB2（未定義は NaN）
    """
    N = x.size
    h = np.bincount(np.asarray(x, dtype=int).ravel(), minlength=L)   # 手順 1：度数 h_x
    n0, n1, c0, c1 = (np.zeros(L) for _ in range(4))
    mu0, mu1, sB = (np.full(L, np.nan) for _ in range(3))   # μ0(0)，σB²(0) は NaN
    n1[0] = N                                    # 手順 2：τ=0 の初期化
    c1[0] = np.arange(L) @ h
    mu1[0] = c1[0] / n1[0]
    for tau in range(1, L):                      # 手順 3：τ=1,2,...,L-1
        n0[tau] = n0[tau - 1] + h[tau - 1]
        n1[tau] = n1[tau - 1] - h[tau - 1]
        c0[tau] = c0[tau - 1] + (tau - 1) * h[tau - 1]
        c1[tau] = c1[tau - 1] - (tau - 1) * h[tau - 1]
        if n0[tau] != 0:
            mu0[tau] = c0[tau] / n0[tau]
        if n1[tau] != 0:
            mu1[tau] = c1[tau] / n1[tau]
        sB[tau] = n0[tau] * n1[tau] * (mu0[tau] - mu1[tau]) ** 2 / N**2   # 平均が NaN なら NaN
    tauStar = int(np.nanargmax(sB))              # 手順 4（NaN を無視して最大を探す）
    tbl = dict(tau=np.arange(L), h=h, n0=n0, n1=n1, mu0=mu0, mu1=mu1, sigmaB2=sB)
    return tauStar, tbl

## 大津法が有効な例
暗い背景に星形の物体があるノイズを含む画像（背景 40，物体 110，ノイズの標準偏差 18）。階調の中央 $\tau=128$ では物体がほとんど消えるが，大津法ではヒストグラムから適切な閾値が選ばれる。

星形の多角形は scikit-image の `draw.polygon2mask` で二値画像にする。ノイズは NumPy の正規乱数で作るので，値はスライド（MATLAB 版）と一致しない。

In [ ]:
rng = np.random.default_rng(0)
N = 128
t = np.linspace(0, 2 * np.pi, 11)[:-1]
rad = np.tile([48, 20], 5)                       # 星の外側・内側の半径
ctr = (N - 1) / 2                                # 画像の中心
verts = np.column_stack([ctr + rad * np.sin(t - np.pi / 2), ctr + rad * np.cos(t - np.pi / 2)])   # 頂点の (行, 列)
star = polygon2mask((N, N), verts)
S = 40 + 70 * star + 18 * rng.standard_normal((N, N))   # 背景 40，物体 110，ノイズ σ=18
S = np.clip(np.round(S), 0, 255)
tauS, _ = otsu(S, 256)                           # 大津法の閾値 τ*
print("τ* =", tauS)
print([tauS, int(threshold_otsu(S.astype(np.uint8))) + 1])   # threshold_otsu との関係を確認

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 3.4), layout="constrained")
show(axs[0], S / 255, "ノイズを含む画像")
show(axs[1], (S >= 128).astype(float), r"$\tau=128$")
show(axs[2], (S >= tauS).astype(float), rf"大津法（$\tau^\star={tauS}$）")
plt.show()

度数分布 $h_x$ と 2 つの閾値。大津法の閾値 $\tau^\star$ は背景と物体の 2 つの山の間に置かれる。

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.4), layout="constrained")
ax.hist(S.ravel(), np.arange(-0.5, 256), color=ccool, label="$h_x$")
ax.axvline(128, ls="--", color=cgray, lw=2, label=r"$\tau=128$")
ax.axvline(tauS, ls="-", color=cwarm, lw=2.5, label=rf"$\tau^\star={tauS}$")
ax.legend(loc="upper right")
ax.set(xlim=(0, 255), xlabel="$x$", ylabel="$h_x$")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 大津法の例題（教科書）
$L=8$ の $6\times8$ 配列（教科書の例題「ヒストグラム均等化」の配列，第4回と同じ）に大津法を適用する。教科書の表（クラス間分散）と一致し，$\tau^\star=3$ となる。

In [ ]:
Xe = np.array([[2, 2, 3, 1, 2, 4, 1, 1], [3, 4, 4, 3, 2, 3, 4, 2], [4, 4, 4, 3, 4, 5, 5, 4],
               [0, 4, 6, 4, 2, 3, 4, 2], [2, 2, 2, 3, 5, 1, 3, 3], [2, 1, 4, 3, 4, 1, 6, 0]])
L, Ne = 8, Xe.size
tauE, tbl = otsu(Xe, L)
print("τ* =", tauE)


def fmt2(v):
    """小数点以下 2 桁の文字列にする（未定義の値は教科書どおり NaN と書く）．"""
    return ["NaN" if np.isnan(a) else f"{a:.2f}" for a in v]


for key, v in tbl.items():
    cells = fmt2(v) if key.startswith(("mu", "sigma")) else [str(int(a)) for a in v]
    print(f"{key:>8s}:", " ".join(f"{s:>5s}" for s in cells))

定義どおりに（黒・白クラスの画素を直接取り出して）計算しても同じ値になることを確かめる。

In [ ]:
sBdirect = np.full(L, np.nan)
for tau in range(1, L):
    blk, wht = Xe[Xe < tau], Xe[Xe >= tau]
    if blk.size and wht.size:                    # 空のクラスの平均は NaN
        sBdirect[tau] = blk.size * wht.size * (blk.mean() - wht.mean()) ** 2 / Ne**2
print(np.allclose(sBdirect, tbl["sigmaB2"], rtol=0, atol=1e-12, equal_nan=True))
Ye = (Xe >= tauE).astype(int)
print(Ye)

スライドの表の値と一致することを確かめる。

In [ ]:
assert tauE == 3
assert list(tbl["n0"]) == [0, 2, 8, 19, 29, 43, 46, 48] and list(tbl["n1"]) == [48, 46, 40, 29, 19, 5, 2, 0]
assert fmt2(tbl["mu0"]) == ["NaN", "0.00", "0.75", "1.47", "2.00", "2.65", "2.80", "2.94"]
assert fmt2(tbl["mu1"]) == ["2.94", "3.07", "3.38", "3.90", "4.37", "5.40", "6.00", "NaN"]
assert fmt2(tbl["sigmaB2"]) == ["NaN", "0.38", "0.96", "1.40", "1.34", "0.71", "0.41", "NaN"]
assert np.array_equal(Ye, [[0, 0, 1, 0, 0, 1, 0, 0], [1, 1, 1, 1, 0, 1, 1, 0], [1, 1, 1, 1, 1, 1, 1, 1],
                           [0, 1, 1, 1, 0, 1, 1, 0], [0, 0, 0, 1, 1, 0, 1, 1], [0, 0, 1, 1, 1, 0, 1, 0]])

## 大津法による処理画像
教科書の図 2.7(b) は教科書の画像（$96\times96$）で $\tau^\star=84$。ここでは同じ画像 msipimg08 を $256\times256$ に縮小したものに同じ手順を適用する（縮小のため閾値は少し異なる）。

In [ ]:
tauC, _ = otsu(X, 256)
print("τ* =", tauC)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6), layout="constrained")
show(axs[0], (X >= 128).astype(float), r"$\tau=128$")
show(axs[1], (X >= tauC).astype(float), rf"大津法（$\tau^\star={tauC}$）")
plt.show()

## パターンニング
$4\times4$ の 17 種類の二値パターン（白 0 個〜16 個）を用意し，縮小した画像の各画素を対応するパターンに置き換える。パターンは次のディザ配列の順に白を増やして作る。

In [ ]:
Td = np.array([[0, 128, 32, 160], [192, 64, 224, 96], [48, 176, 16, 144], [240, 112, 208, 80]])   # 教科書のディザ配列
rnk = Td // 16                                   # 0〜15 の順位
pat = [(rnk < k).astype(float) for k in range(17)]   # 白の画素が k 個
tiles = np.ones((4 * 2 + 1, (4 + 1) * 17 + 1))   # パターン一覧（灰色の枠）
for k in range(17):
    tiles[1:5, 5 * k + 1:5 * k + 5] = pat[k]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 1.4), layout="constrained")
ax.imshow(tiles, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
ax.set_axis_off()
for k in range(17):
    ax.add_patch(plt.Rectangle((5 * k + 0.5, 0.5), 4, 4, fill=False, ec=cgray, lw=1))
    ax.text(5 * k + 2.5, 6.5, str(k), ha="center", va="center", fontsize=9)
plt.show()

画像を $4\times4$ 画素ごとの平均で $64\times64$ に縮小し（scikit-image の `downscale_local_mean`），各画素の明るさを 0〜16 のレベルにしてパターンに置き換える。

In [ ]:
Xs = downscale_local_mean(X, (4, 4))                               # 64×64 に縮小
lev = np.floor(Xs / 255 * 16 + 0.5).astype(int)      # 0〜16 のレベル
P = np.zeros_like(X)
for r in range(Xs.shape[0]):
    for c in range(Xs.shape[1]):
        P[4 * r:4 * r + 4, 4 * c:4 * c + 4] = pat[lev[r, c]]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6), layout="constrained")
show(axs[0], X / 255, "原画像")
show(axs[1], P, "パターンニング")
plt.show()

## ディザリング：前回スライドの数値例
$8\times8$ の入力をディザ配列（$4\times4$ を並べたもの）と画素ごとに比較して二値化する（$x\ge$ 閾値なら 1）。

In [ ]:
Xin = np.array([[12, 51, 14, 31, 16, 50, 60, 70], [30, 55, 23, 100, 13, 99, 79, 83],
                [77, 65, 199, 203, 202, 200, 85, 99], [66, 58, 43, 11, 15, 65, 89, 91],
                [87, 81, 64, 24, 98, 56, 80, 19], [41, 98, 31, 30, 18, 16, 9, 0],
                [78, 43, 51, 61, 45, 53, 87, 15], [64, 53, 41, 43, 61, 13, 71, 115]])
T8 = np.tile(Td, (2, 2))
Yd = (Xin >= T8).astype(int)
print(Yd)
assert np.array_equal(Yd, [[1, 0, 0, 0, 1, 0, 1, 0], [0, 0, 0, 1, 0, 1, 0, 0], [1, 0, 1, 1, 1, 1, 1, 0],
                           [0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 0, 0, 0, 0, 0],
                           [1, 0, 1, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 1]])   # スライドの値と一致

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6), layout="constrained")
show(axs[0], Xin / 255, "入力")
show(axs[1], Yd.astype(float), "ディザリングの出力")
plt.show()

スライドの説明に使う 2 画素（左上とその右）の比較。

In [ ]:
rel = ["<", "≥"]
for k in range(2):
    print(f"{Xin[0, k]} {rel[Yd[0, k]]} {T8[0, k]}  →  {Yd[0, k]}")

## ディザリングの処理画像

In [ ]:
Tfull = np.tile(Td, (X.shape[0] // 4, X.shape[1] // 4))
Dth = (X >= Tfull).astype(float)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6), layout="constrained")
show(axs[0], X / 255, "原画像")
show(axs[1], Dth, "ディザリング")
plt.show()

## 誤差拡散（Floyd & Steinberg）
左上から右へ（ラスタ走査）二値化し，量子化誤差 $e$ を右 7/16，左下 3/16，下 5/16，右下 1/16 の割合で未処理の画素に配る。

In [ ]:
def floydsteinberg(x, L=1):
    """フロイド・スタインバーグの誤差拡散（x は 0〜1，L+1 階調に量子化，既定は二値）．"""
    nr, nc = x.shape
    w = np.zeros((nr + 1, nc + 2))               # 右・下・左に余白
    w[:nr, 1:nc + 1] = x
    w = w.tolist()                               # 1 画素ずつの処理はリストの方が速い
    y = np.zeros((nr, nc))
    for r in range(nr):
        for c in range(nc):
            v = w[r][c + 1]                      # 誤差を加えた値
            q = min(max(math.floor(v * L + 0.5), 0), L) / L   # 四捨五入で量子化（二値なら閾値 0.5）
            y[r, c] = q
            e = v - q                            # 量子化誤差
            w[r][c + 2] += e * 7 / 16
            w[r + 1][c] += e * 3 / 16
            w[r + 1][c + 1] += e * 5 / 16
            w[r + 1][c + 2] += e * 1 / 16
    return y


E = floydsteinberg(X / 255)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6), layout="constrained")
show(axs[0], X / 255, "原画像")
show(axs[1], E, "誤差拡散")
plt.show()

局所的な白画素の割合は元の明るさに近い（$8\times8$ 平均の比較）。

In [ ]:
Xb8 = downscale_local_mean(X / 255, (8, 8))
loc = [np.mean(np.abs(downscale_local_mean(E, (8, 8)) - Xb8)),
       np.mean(np.abs(downscale_local_mean(Dth, (8, 8)) - Xb8)),
       np.mean(np.abs(downscale_local_mean((X >= 128).astype(float), (8, 8)) - Xb8))]
print("誤差拡散 {:.3f}，ディザリング {:.3f}，閾値処理 {:.3f}".format(*loc))

数値例：一様な明るさ 0.25 の $8\times8$ 画像では，白画素の割合がほぼ 1/4 になる（端では誤差を配りきれないので少しずれる）。

In [ ]:
E25 = floydsteinberg(0.25 * np.ones((8, 8))).astype(int)
print(E25)
print("白画素", E25.sum(), "個，割合", E25.mean())
assert E25.sum() == 14 and np.array_equal(E25, [[0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 1, 0, 1, 0, 1],
                                                [0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 1, 0, 1, 0, 1],
                                                [0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 1, 0, 1, 0, 0],
                                                [0, 0, 0, 0, 0, 0, 1, 0], [0, 1, 0, 1, 0, 0, 0, 0]])   # スライドの値と一致

## カラー画像への応用：24 bpp → 8 bpp
R, G, B を 3, 3, 2 bit（計 8 bpp）に減らす。線形量子化と成分ごとの誤差拡散を比べる。写真は msipimg01（海岸，$256\times256$ に縮小）。空のなめらかな階調で，線形量子化の擬似輪郭と誤差拡散の効果がはっきり見える。

In [ ]:
Xc = img_as_float(vie.msipimg(1, 256))
bits = [3, 3, 2]
Xlq = np.zeros_like(Xc)
Xed = np.zeros_like(Xc)
for k in range(3):
    Lk = 2 ** bits[k] - 1
    Xlq[:, :, k] = np.floor(Xc[:, :, k] * Lk + 0.5) / Lk         # 線形量子化
    Xed[:, :, k] = floydsteinberg(Xc[:, :, k], Lk)          # 多値の誤差拡散

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], Xc, "原画像（24 bpp）")
show(axs[1], Xlq, "線形量子化（8 bpp）")
show(axs[2], Xed, "誤差拡散（8 bpp）")
for ax in axs:
    ax.add_patch(plt.Rectangle((139.5, -0.5), 60, 60, fill=False, ec=cwarm, lw=1.5))
plt.show()

空の一部（橙の枠）を拡大して比べる。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6), layout="constrained")
show(axs[0], Xlq[0:60, 140:200], "線形量子化（拡大）")
show(axs[1], Xed[0:60, 140:200], "誤差拡散（拡大）")
plt.show()

## 順序統計フィルタ
近傍領域 $\mathcal{N}_\mathrm{f}$（$3\times3$）の画素値を昇順に並べ，$K$ 番目の値を出力する（教科書 3.3 節）。前回スライドの例で，中央値（$K=(|\mathcal{N}_\mathrm{f}|+1)/2=5$），最小値（$K=1$），最大値（$K=|\mathcal{N}_\mathrm{f}|=9$）を求める。

In [ ]:
B = np.array([[10, 20, 20], [20, 15, 20], [20, 25, 100]])
v = np.sort(B.ravel())
print("昇順:", v)
K = [(v.size + 1) // 2, 1, v.size]               # 中央値，最小値，最大値の順位
vals = v[np.array(K) - 1]                        # ord(・,K)（K 番目は添字 K-1）
print("中央値，最小値，最大値:", vals)
print(np.array_equal(vals, [np.median(B), B.min(), B.max()]))
assert list(v) == [10, 15, 20, 20, 20, 20, 20, 25, 100] and list(vals) == [20, 10, 100]   # スライドの値と一致

二値の例。二値画像では，最小値フィルタは近傍の論理積，最大値フィルタは近傍の論理和になる。昇順に並べた列の $K$ 番目を示す（ここでは [ ] で囲む。スライドでは教科書の例題「中央値フィルタ」の解答と同じく下線を付ける）。

In [ ]:
def ordlist(v, K):
    """昇順に並べた列 v を「, 」区切りの文字列にし，K 番目を [ ] で囲む．"""
    c = [str(a) for a in v]
    c[K - 1] = "[" + c[K - 1] + "]"
    return ", ".join(c)


Bb = np.array([[0, 1, 1], [1, 0, 1], [1, 1, 1]])
vb = np.sort(Bb.ravel())
valsb = [vb[0], vb[-1]]
print("昇順:", vb)
print("最小値（K=1）:", ordlist(vb, 1), "→", valsb[0])
print("最大値（K=9）:", ordlist(vb, vb.size), "→", valsb[1])
print(valsb == [int(Bb.all()), int(Bb.any())])

2 つの配列を，ます目と数値で描く（前回の演習課題の配列の図を参考）。中央の対象画素を橙で示す。

In [ ]:
def drawgrid(ax, A, mask, fc):
    """配列 A をます目と数値で描く（mask の画素は色 fc で塗る）．"""
    nr, nc = A.shape
    for r in range(nr):
        for c in range(nc):
            ax.add_patch(plt.Rectangle((c, nr - 1 - r), 1, 1, fc=fc if mask[r, c] else "w", ec="k", lw=1.2))
            ax.text(c + 0.5, nr - 1 - r + 0.5, str(A[r, c]), ha="center", va="center", fontsize=16)
    ax.set(xlim=(-0.03, nc + 0.03), ylim=(-0.03, nr + 0.03), aspect="equal")
    ax.set_axis_off()


ctrm = np.zeros((3, 3), dtype=bool)
ctrm[1, 1] = True
fig, axs = plt.subplots(1, 2, figsize=(6, 2.8), layout="constrained")
drawgrid(axs[0], B, ctrm, 0.3 * np.array(cwarm) + 0.7)
drawgrid(axs[1], Bb, ctrm, 0.3 * np.array(cwarm) + 0.7)
plt.show()

## 例題「最大値／最小値フィルタ」（教科書 3.3.2 項）
教科書の例題「矩形フィルタ」の配列 $\mathsf{x}$ に $3\times3$ の最大値フィルタと最小値フィルタを施す。周囲の値はすべて零値とする（`mode="constant"`）。教科書の解答と一致する。

In [ ]:
Xm = np.array([[18, 9, 9, 9], [27, 9, 9, 9], [36, 9, 9, 9]])
Tmax = ndimage.maximum_filter(Xm, size=3, mode="constant", cval=0)   # K=|N_f|=9
Tmin = ndimage.minimum_filter(Xm, size=3, mode="constant", cval=0)   # K=1
Tmed = ndimage.median_filter(Xm, size=3, mode="constant", cval=0)    # 参考：例題「中央値フィルタ」（K=5）
print("最大値フィルタ\n", Tmax)
print("最小値フィルタ\n", Tmin)
print("中央値フィルタ\n", Tmed)
assert (np.array_equal(Tmax, [[27, 27, 9, 9], [36, 36, 9, 9], [36, 36, 9, 9]])
        and np.array_equal(Tmin, [[0, 0, 0, 0], [0, 9, 9, 0], [0, 0, 0, 0]])
        and np.array_equal(Tmed, [[0, 9, 9, 0], [9, 9, 9, 9], [0, 9, 9, 0]]))

## モルフォロジー処理
星形の二値画像にごま塩ノイズと小さな穴を加え，$3\times3$ の平坦な正方形の構造化要素 $\mathsf{h}$ で収縮（最小値フィルタ），膨張（最大値フィルタ），オープニング（収縮の後に膨張），クロージング（膨張の後に収縮）を施す。

In [ ]:
rng = np.random.default_rng(2)
A = star.copy()
A[rng.random((N, N)) < 0.02] = True              # 背景の白い点（ごま）
A[star & (rng.random((N, N)) < 0.03)] = False    # 物体の中の黒い穴（しお）
se = np.ones((3, 3), dtype=bool)


def erode(M):
    return ndimage.binary_erosion(M, se, border_value=1)     # 周囲は 1（+∞）


def dilate(M):
    return ndimage.binary_dilation(M, se, border_value=0)    # 周囲は 0（-∞）


Aer, Adi = erode(A), dilate(A)
Aop, Acl = dilate(erode(A)), erode(dilate(A))

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(12, 2.8), layout="constrained")
for ax, M, ttl in zip(axs, (A, Aer, Adi, Aop, Acl), ("原画像", "収縮", "膨張", "オープニング", "クロージング")):
    show(ax, M.astype(float), ttl)
plt.show()

最小値フィルタ・最大値フィルタと一致することを確かめる。教科書どおり，収縮では周囲に $+\infty$（二値では 1），膨張では $-\infty$（二値では 0）を仮定する。

In [ ]:
Amin = ndimage.minimum_filter(np.pad(A, 1, constant_values=True), size=3)[1:-1, 1:-1]
Amax = ndimage.maximum_filter(np.pad(A, 1, constant_values=False), size=3)[1:-1, 1:-1]
same = [np.array_equal(Aer, Amin), np.array_equal(Adi, Amax)]
print(same)

残った白い点・穴の数（連結成分，8 近傍）を数える。

In [ ]:
def cnt(M):
    """白領域の数と穴の数（黒領域の数から背景の 1 個を引く）．"""
    return [ndimage.label(M, se)[1], ndimage.label(~M, se)[1] - 1]


counts = np.array([cnt(A), cnt(Aop), cnt(Acl)])  # 行：原画像，オープニング，クロージング／列：白領域数，穴の数
print(counts)

## まとめ
- 大津法はクラス間分散を最大にする閾値を自動で選ぶ（未定義の平均値は NaN）
- ハーフトーニング（パターンニング，ディザリング，誤差拡散）は二値で階調を擬似表現する
- モルフォロジー処理の収縮・膨張は最小値・最大値フィルタで実現でき，その組合せがオープニング・クロージング

© Copyright, Shogo MURAMATSU, All rights reserved.